In [1]:
import pandas as pd
import numpy as np
import psycopg2
import sqlalchemy as db
from sqlalchemy import create_engine
import yaml

In [2]:
with open("../config.yml", "r") as f:
    config = yaml.safe_load(f)
    config_mensajeria = config["MENSAJERIA_OLTP"]
    config_etl = config["ETL_PROCESS"]

# Construct the database URL
url_mensajeria = (
    f"{config_mensajeria['drivername']}://{config_mensajeria['user']}:{config_mensajeria['password']}@{config_mensajeria['host']}:"
    f"{config_mensajeria['port']}/{config_mensajeria['dbname']}"
)
url_etl = (
    f"{config_etl['drivername']}://{config_etl['user']}:{config_etl['password']}@{config_etl['host']}:"
    f"{config_etl['port']}/{config_etl['dbname']}"
)
# Create the SQLAlchemy Engine
mensajeria = create_engine(url_mensajeria)
etl_conn = create_engine(url_etl)

In [3]:
estadosservicio = pd.read_sql_table('mensajeria_estadosservicio', mensajeria)
estado = pd.read_sql_table('mensajeria_estado',mensajeria)
servicio = pd.read_sql_table('mensajeria_servicio',mensajeria)
dim_fechahora = pd.read_sql_table('dim_fechahora',etl_conn)

In [4]:
estadosservicio["fecha_hora"] = pd.to_datetime(
    (
        estadosservicio["fecha"].astype(str) + " " + estadosservicio["hora"].astype(str)
    ).str[:19]
)

estadosservicio.head()

,id,fecha,hora,foto,observaciones,estado_id,servicio_id,es_prueba,foto_binary,fecha_hora
0,1014,2024-01-29,01:13:32,foto,4 tubos,4,226,False,None,2024-01-29 01:13:32
1,1484,2024-01-30,18:45:12,foto,Demora,3,79,True,None,2024-01-30 18:45:12
2,2829,2024-02-06,11:34:04,foto,Compra exitosa,5,613,False,None,2024-02-06 11:34:04
3,1888,2024-02-01,14:50:39,foto,Zzxzz,4,376,False,None,2024-02-01 14:50:39
4,32312,2024-04-06,16:11:21,foto,No,3,7164,True,None,2024-04-06 16:11:21


In [5]:
mapa_estados = {1: "iniciado", 2: "asignado", 4: "recogido", 5: "entregado", 6: "cerrado"}

In [6]:
agg = (
    estadosservicio[estadosservicio["estado_id"].isin(mapa_estados.keys())]
    .groupby(["servicio_id", "estado_id"])["fecha_hora"]
    .min()
    .reset_index()
)

In [7]:
agg["estado"] = agg["estado_id"].map(mapa_estados)

In [8]:
estados_wide = agg.pivot(
    index="servicio_id", columns="estado", values="fecha_hora"
).reset_index()

In [9]:
hecho_servicio = servicio[["id"]].rename(columns={"id": "servicio_id"})
hecho_servicio = hecho_servicio.merge(estados_wide, on="servicio_id", how="left")

In [10]:
# Calcular duraciones en minutos ANTES de convertir las fechas en llaves
hecho_servicio["duracion_iniciado_asignado_min"] = (
    (hecho_servicio["asignado"] - hecho_servicio["iniciado"]).dt.total_seconds() / 60
)
hecho_servicio["duracion_asignado_recogido_min"] = (
    (hecho_servicio["recogido"] - hecho_servicio["asignado"]).dt.total_seconds() / 60
)
hecho_servicio["duracion_recogido_entregado_min"] = (
    (hecho_servicio["entregado"] - hecho_servicio["recogido"]).dt.total_seconds() / 60
)
hecho_servicio["duracion_entregado_cerrado_min"] = (
    (hecho_servicio["cerrado"] - hecho_servicio["entregado"]).dt.total_seconds() / 60
)
hecho_servicio["duracion_total_min"] = (
    (hecho_servicio["cerrado"] - hecho_servicio["iniciado"]).dt.total_seconds() / 60
)

hecho_servicio[[
    "servicio_id",
    "duracion_iniciado_asignado_min",
    "duracion_asignado_recogido_min",
    "duracion_recogido_entregado_min",
    "duracion_entregado_cerrado_min",
    "duracion_total_min",
]].head()

,servicio_id,duracion_iniciado_asignado_min,duracion_asignado_recogido_min,duracion_recogido_entregado_min,duracion_entregado_cerrado_min,duracion_total_min
0,34,NaN,NaN,NaN,NaN,NaN
1,35,3084.9,302.166667,56500.266667,NaN,NaN
2,36,NaN,NaN,NaN,NaN,NaN
3,41,NaN,NaN,NaN,NaN,NaN
4,42,NaN,NaN,NaN,NaN,NaN


In [11]:
KEY_FECHA_DESCONOCIDA = 127010  # fila "no aplica" que ya existe en dim_fechahora

KEY_FECHA_DESCONOCIDA = 127010  # fila "no aplica" que ya existe en dim_fechahora

dim_fechahora["fecha_hora"] = pd.to_datetime(dim_fechahora["fecha_hora"])
# Nos quedamos solo con las filas REALES (con fecha) para el lookup,
# así evitamos el emparejamiento accidental NaT == NaT
dim_fh_lookup = dim_fechahora.loc[
    dim_fechahora["fecha_hora"].notna(), ["key_dim_fechahora", "fecha_hora"]
]

for col in ["iniciado", "asignado", "recogido", "entregado", "cerrado"]:
    # por si acaso, forzamos también la columna del hito a datetime
    hecho_servicio[col] = pd.to_datetime(hecho_servicio[col], errors="coerce")

    hecho_servicio = hecho_servicio.merge(
        dim_fh_lookup, left_on=col, right_on="fecha_hora", how="left"
    )
    hecho_servicio = hecho_servicio.rename(
        columns={"key_dim_fechahora": f"fk_fecha_{col}"}
    )
    hecho_servicio = hecho_servicio.drop(columns=["fecha_hora", col])

    # Reemplazar cualquier NULL (hito no alcanzado O timestamp sin match)
    # por la key de "fecha desconocida / no aplica"
    hecho_servicio[f"fk_fecha_{col}"] = (
        hecho_servicio[f"fk_fecha_{col}"].fillna(KEY_FECHA_DESCONOCIDA).astype("Int64")
    )

hecho_servicio.head()

,servicio_id,duracion_iniciado_asignado_min,duracion_asignado_recogido_min,duracion_recogido_entregado_min,duracion_entregado_cerrado_min,duracion_total_min,fk_fecha_iniciado,fk_fecha_asignado,fk_fecha_recogido,fk_fecha_entregado,fk_fecha_cerrado
0,34,NaN,NaN,NaN,NaN,NaN,90,127010,127010,127010,127010
1,35,3084.9,302.166667,56500.266667,NaN,NaN,91,92,94,163,127010
2,36,NaN,NaN,NaN,NaN,NaN,93,127010,127010,127010,127010
3,41,NaN,NaN,NaN,NaN,NaN,104,127010,127010,127010,127010
4,42,NaN,NaN,NaN,NaN,NaN,105,127010,127010,127010,127010


In [12]:
fk_cols = [
    "fk_fecha_iniciado",
    "fk_fecha_asignado",
    "fk_fecha_recogido",
    "fk_fecha_entregado",
    "fk_fecha_cerrado",
]

for col in fk_cols:
    hecho_servicio[col] = hecho_servicio[col].astype("Int64")

In [13]:
for col in fk_cols:
    total = hecho_servicio[col].notna().sum()
    print(f"{col}: {total} de {len(hecho_servicio)} servicios con este hito")

hecho_servicio.head()

fk_fecha_iniciado: 28430 de 28430 servicios con este hito
fk_fecha_asignado: 28430 de 28430 servicios con este hito
fk_fecha_recogido: 28430 de 28430 servicios con este hito
fk_fecha_entregado: 28430 de 28430 servicios con este hito
fk_fecha_cerrado: 28430 de 28430 servicios con este hito


,servicio_id,duracion_iniciado_asignado_min,duracion_asignado_recogido_min,duracion_recogido_entregado_min,duracion_entregado_cerrado_min,duracion_total_min,fk_fecha_iniciado,fk_fecha_asignado,fk_fecha_recogido,fk_fecha_entregado,fk_fecha_cerrado
0,34,NaN,NaN,NaN,NaN,NaN,90,127010,127010,127010,127010
1,35,3084.9,302.166667,56500.266667,NaN,NaN,91,92,94,163,127010
2,36,NaN,NaN,NaN,NaN,NaN,93,127010,127010,127010,127010
3,41,NaN,NaN,NaN,NaN,NaN,104,127010,127010,127010,127010
4,42,NaN,NaN,NaN,NaN,NaN,105,127010,127010,127010,127010


In [14]:
hecho_servicio.isnull().sum()

servicio_id                            0
duracion_iniciado_asignado_min       729
duracion_asignado_recogido_min      1414
duracion_recogido_entregado_min     1479
duracion_entregado_cerrado_min     20141
duracion_total_min                 20140
fk_fecha_iniciado                      0
fk_fecha_asignado                      0
fk_fecha_recogido                      0
fk_fecha_entregado                     0
fk_fecha_cerrado                       0
dtype: int64

In [15]:
dim_cliente = pd.read_sql_table('dim_cliente',etl_conn)

In [16]:
servicio[["id", "cliente_id"]].head()

,id,cliente_id
0,34,5
1,35,5
2,36,5
3,41,5
4,42,5


In [17]:
hecho_servicio = hecho_servicio.merge(
    servicio[["id", "cliente_id"]].rename(columns={"id": "servicio_id"}),
    on="servicio_id",
    how="left",
)

hecho_servicio.head()

,servicio_id,duracion_iniciado_asignado_min,duracion_asignado_recogido_min,duracion_recogido_entregado_min,duracion_entregado_cerrado_min,duracion_total_min,fk_fecha_iniciado,fk_fecha_asignado,fk_fecha_recogido,fk_fecha_entregado,fk_fecha_cerrado,cliente_id
0,34,NaN,NaN,NaN,NaN,NaN,90,127010,127010,127010,127010,5
1,35,3084.9,302.166667,56500.266667,NaN,NaN,91,92,94,163,127010,5
2,36,NaN,NaN,NaN,NaN,NaN,93,127010,127010,127010,127010,5
3,41,NaN,NaN,NaN,NaN,NaN,104,127010,127010,127010,127010,5
4,42,NaN,NaN,NaN,NaN,NaN,105,127010,127010,127010,127010,5


In [18]:
hecho_servicio = hecho_servicio.merge(
    dim_cliente[["cliente_key", "cliente_id"]], on="cliente_id", how="left"
)

In [19]:
hecho_servicio = hecho_servicio.rename(columns={"cliente_key": "fk_cliente"})
hecho_servicio = hecho_servicio.drop(columns=["cliente_id"])

In [20]:
hecho_servicio["fk_cliente"] = hecho_servicio["fk_cliente"].astype("Int64")

hecho_servicio.head()

,servicio_id,duracion_iniciado_asignado_min,duracion_asignado_recogido_min,duracion_recogido_entregado_min,duracion_entregado_cerrado_min,duracion_total_min,fk_fecha_iniciado,fk_fecha_asignado,fk_fecha_recogido,fk_fecha_entregado,fk_fecha_cerrado,fk_cliente
0,34,NaN,NaN,NaN,NaN,NaN,90,127010,127010,127010,127010,5
1,35,3084.9,302.166667,56500.266667,NaN,NaN,91,92,94,163,127010,5
2,36,NaN,NaN,NaN,NaN,NaN,93,127010,127010,127010,127010,5
3,41,NaN,NaN,NaN,NaN,NaN,104,127010,127010,127010,127010,5
4,42,NaN,NaN,NaN,NaN,NaN,105,127010,127010,127010,127010,5


In [21]:
dim_mensajero = pd.read_sql_table('dim_mensajero', etl_conn)

In [22]:
mensajero_servicio = servicio[["id", "mensajero_id"]].rename(
    columns={"id": "servicio_id"}
)
mensajero_servicio["mensajero_id"] = (
    mensajero_servicio["mensajero_id"].fillna(-1).astype("int64")
)

In [23]:
hecho_servicio = hecho_servicio.merge(
    mensajero_servicio, on="servicio_id", how="left"
)

hecho_servicio = hecho_servicio.merge(
    dim_mensajero[["mensajero_key", "mensajero_id"]], on="mensajero_id", how="left"
)

hecho_servicio = hecho_servicio.rename(columns={"mensajero_key": "fk_mensajero"})
hecho_servicio = hecho_servicio.drop(columns=["mensajero_id"])

hecho_servicio["fk_mensajero"] = hecho_servicio["fk_mensajero"].astype("Int64")

In [24]:
hecho_servicio.head()

,servicio_id,duracion_iniciado_asignado_min,duracion_asignado_recogido_min,duracion_recogido_entregado_min,duracion_entregado_cerrado_min,duracion_total_min,fk_fecha_iniciado,fk_fecha_asignado,fk_fecha_recogido,fk_fecha_entregado,fk_fecha_cerrado,fk_cliente,fk_mensajero
0,34,NaN,NaN,NaN,NaN,NaN,90,127010,127010,127010,127010,5,-1
1,35,3084.9,302.166667,56500.266667,NaN,NaN,91,92,94,163,127010,5,7
2,36,NaN,NaN,NaN,NaN,NaN,93,127010,127010,127010,127010,5,-1
3,41,NaN,NaN,NaN,NaN,NaN,104,127010,127010,127010,127010,5,-1
4,42,NaN,NaN,NaN,NaN,NaN,105,127010,127010,127010,127010,5,-1


In [25]:
dim_sede = pd.read_sql_table('dim_sede',etl_conn)
clientes_usuario = pd.read_sql_table('clientes_usuarioaquitoy', mensajeria)

In [26]:
usuario_servicio = servicio[["id", "usuario_id"]].rename(columns={"id": "servicio_id"})

In [27]:
usuario_servicio = usuario_servicio.merge(
    clientes_usuario[["id", "sede_id"]].rename(columns={"id": "usuario_id"}),
    on="usuario_id",
    how="left",
)

In [28]:
hecho_servicio = hecho_servicio.merge(
    usuario_servicio[["servicio_id", "sede_id"]], on="servicio_id", how="left"
)

In [29]:
hecho_servicio = hecho_servicio.merge(
    dim_sede[["key_dim_sede", "sede_id"]].rename(columns={"key_dim_sede": "fk_sede"}),
    on="sede_id",
    how="left",
).drop(columns=["sede_id"])

In [30]:
hecho_servicio["fk_sede"] = hecho_servicio["fk_sede"].astype("Int64")

In [31]:
hecho_servicio
# hecho_servicio['fk_cliente'].isnull().sum()

,servicio_id,duracion_iniciado_asignado_min,duracion_asignado_recogido_min,duracion_recogido_entregado_min,duracion_entregado_cerrado_min,duracion_total_min,fk_fecha_iniciado,fk_fecha_asignado,fk_fecha_recogido,fk_fecha_entregado,fk_fecha_cerrado,fk_cliente,fk_mensajero,fk_sede
0,34,NaN,NaN,NaN,NaN,NaN,90,127010,127010,127010,127010,5,-1,24
1,35,3084.900000,302.166667,56500.266667,NaN,NaN,91,92,94,163,127010,5,7,16
2,36,NaN,NaN,NaN,NaN,NaN,93,127010,127010,127010,127010,5,-1,16
3,41,NaN,NaN,NaN,NaN,NaN,104,127010,127010,127010,127010,5,-1,15
4,42,NaN,NaN,NaN,NaN,NaN,105,127010,127010,127010,127010,5,-1,15
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
28425,28267,8.883333,32.750000,14.200000,NaN,NaN,126008,126025,126088,126112,127010,11,8,7
28426,28363,7.966667,5.850000,73.766667,NaN,NaN,126443,126456,126469,126572,127010,11,20,7
28427,28403,3.866667,52.250000,38.250000,NaN,NaN,126662,126663,126685,126720,127010,5,28,49
28428,28438,1.533333,22.016667,20.016667,0.616667,44.183333,126814,126816,126852,126872,126873,11,37,4


In [32]:
hecho_servicio.to_sql("fact_servicio", etl_conn, if_exists="replace", index=False)

430